# Ethio Speech AI — teammate Colab demo

Start with `COLAB_GUIDE.md` from the same ZIP. Upload `ethio-speech-ai-colab.zip` to
`MyDrive/ethio_speech_ai/inputs/` in **your own Drive**. Choose a T4 GPU and enable
notebook access to your `HF_TOKEN` secret. Run cells **1–5** in order; speak and
review the eight questions; export feedback in **6**; stop servers in **7**.

Selected stack: Hohe FP16 → local bank search → Gemma 4 E4B Q4_K_M through
llama.cpp → OmniVoice 16 steps. Automatic endpointing is **800 ms**. The package
already includes the 41-document index and three warmup recordings. No previous
results, source DOCX folders or patch scripts are needed. Model weights download
on first setup/startup.

The reference documents are working test material, not verified current bank policy.
The demo does not access accounts or execute transactions. One complete top passage
must fit the 384-token input budget; otherwise it abstains. Test source relevance,
answer correctness and speech quality separately. Colab uses your Google runtime
and Drive. The organizational deployment runs on its own GPU server.


## Cell 1 — Paths

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, zipfile, hashlib, shutil, time, secrets
OWNER = input("Your tester name (letters/numbers/_/- only): ").strip().lower()
assert OWNER and all(c.isascii() and (c.isalnum() or c in "_-") for c in OWNER)
TEAM = Path("/content/drive/MyDrive/ethio_speech_ai")
PACKAGE = TEAM / "inputs/ethio-speech-ai-colab.zip"
OUTPUT = TEAM / "results" / OWNER
repo = Path("/content/ethio-speech-ai")
VENV = Path("/content/ethio-speech-venv")
PYTHON = VENV / "bin/python"
PORT = 8765
WORK = Path("/content/llm-engine-tools")

## Cell 2 — Mount Drive, extract and verify the bank package

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
assert PACKAGE.is_file(), f"Upload {PACKAGE} first."
repo.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(PACKAGE) as z:
    for name in z.namelist():
        assert (repo / name).resolve().is_relative_to(repo.resolve()), "Unsafe ZIP path"
    z.extractall(repo)
for name, sha in json.loads((repo / "BUNDLE_MANIFEST.json").read_text()).items():
    assert hashlib.sha256((repo / name).read_bytes()).hexdigest() == sha, name
print("Package verified. Three existing warmup recordings included.")

## Cell 3 — Reuse dependencies/tools and check document retrieval
Checks CUDA and reuses the integrated environment and model caches when present. Adds no embedding model. The final check downloads/reuses only the pinned tokenizer and prints source matches and exact prompt sizes. It does not run LLM, ASR or TTS inference. `fits: False` means the current complete passage will produce an insufficient-evidence reply; do not raise the context limit here.

In [ ]:
env = os.environ.copy()
env.pop("TORCH_LOGS",None)
env.update(HF_HOME="/content/pipeline-hf-cache", USE_TF="0", USE_FLAX="0",
           TOKENIZERS_PARALLELISM="false", PYTHONUNBUFFERED="1",
           PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
for name in ("CUDA_LAUNCH_BLOCKING","PYTHONPATH","HF_HUB_CACHE","TRANSFORMERS_CACHE"):
    env.pop(name,None)
def command(args):
    with subprocess.Popen([str(x) for x in args],env=env,stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=True) as p:
        try:
            for line in p.stdout: print(line,end="",flush=True)
            status=p.wait()
        except BaseException:
            import signal
            os.killpg(p.pid,signal.SIGKILL);p.wait();raise
    if status:raise RuntimeError(f"Setup failed ({status}). Read the actual error above.")
command([sys.executable,"-c","import torch,torchaudio; print(torch.__version__,torchaudio.__version__); assert torch.cuda.is_available(), 'Choose a GPU runtime'; assert torch.__version__.split('+')[0]==torchaudio.__version__.split('+')[0]; print(torch.cuda.get_device_name(0))"])
if not PYTHON.exists():
    command([sys.executable,"-m","venv","--without-pip","--system-site-packages",VENV])
command([sys.executable,"-m","pip","--python",PYTHON,"install","-r",repo/"requirements/integrated-colab.txt"])
command([sys.executable,"-m","pip","--python",PYTHON,"install","--no-deps","omnivoice==0.2.1"])
command([PYTHON,"-c","import sys; sys.path.insert(0,"+repr(str(repo/'src'))+"); from amharic_s2s.llm_next import imports; tf=imports(preserve_audio=True); from omnivoice import OmniVoice; from transformers import AutoModelForCTC,HiggsAudioV2TokenizerModel; import librosa,soundfile,torch,torchaudio; print('Shared imports ready:',tf.__version__)"])
cache=Path(env["HF_HOME"])/"hub"
cache.mkdir(parents=True,exist_ok=True)
selected=["models--bartowski--google_gemma-4-E4B-it-GGUF","models--snapwre--hohe-asr-amharic","models--google--gemma-4-E4B-it",
          "models--gheero-Leyu--amharic-omnivoice-tts","models--k2-fsa--OmniVoice"]
old_hubs=[p/'hub' for p in Path('/content').glob('*hf-cache*') if p != cache.parent]
old_hubs.append(Path('/root/.cache/huggingface/hub'))
for name in selected:
    dest=cache/name
    if dest.exists() or dest.is_symlink(): continue
    for hub in old_hubs:
        source=hub/name
        if source.is_dir() and not source.is_symlink():
            dest.symlink_to(source.resolve(),target_is_directory=True)
            print('Reusing cache:',source); break
from google.colab import userdata
try: token=userdata.get("HF_TOKEN")
except Exception: token=None
if isinstance(token,str) and token.strip(): env["HF_TOKEN"]=token.strip()
del token
assert env.get("HF_TOKEN"), "Enable access to your HF_TOKEN secret and rerun cell 3."
print("Free local disk GiB:",round(shutil.disk_usage('/content').free/2**30,1))
command([sys.executable,"-m","pip","--python",PYTHON,"install","-r",repo/"requirements/browser-api.txt"])
env["PYTHONPATH"]=str(repo/"src")
print("Dependencies ready. Cell 4 loads models once. No model inference has run yet.")
env["AMHARIC_LLM_ENGINE"]="llama_cpp"
env["AMHARIC_CPP_WORK"]=str(WORK)
env["AMHARIC_KNOWLEDGE_INDEX"]=str(repo/"knowledge/bank_v2/bank_index.json")
command([PYTHON,"-m","amharic_s2s.llm_engine_compare","setup","--repo",repo,"--work",WORK])
print("Pinned llama.cpp ready. Run cell 4 once, then cell 5.")
command([PYTHON,"-m","amharic_s2s.bank_pilot","--repo",repo,"--output",OUTPUT/"retrieval_preflight.json","--tokenizer"])


## Cell 4 — Start the selected models once, with the bank index

In [ ]:
if "SERVER" in globals() and SERVER.poll() is None:
    print("Reusing the running model server. Continue to cell 5.")
else:
    if "SERVER" in globals():
        raise RuntimeError("The previous server stopped. Read /content/ethio-speech-server.log before restarting the runtime.")
    import socket
    with socket.socket() as probe:
        assert probe.connect_ex(("127.0.0.1", PORT)) != 0, "Port already in use; stop the old server first."
    SESSION_DIR=OUTPUT/("session-"+time.strftime("%Y%m%dT%H%M%S")+"-"+secrets.token_hex(3))
    SESSION_DIR.mkdir(parents=True,exist_ok=True)
    env["AMHARIC_CPP_LOG_DIR"]=str(SESSION_DIR/"engine")
    API_TOKEN = secrets.token_urlsafe(32)
    env["AMHARIC_API_TOKEN"] = API_TOKEN
    LOG = Path("/content/ethio-speech-server.log")
    with LOG.open("w", encoding="utf-8") as log:
        SERVER = subprocess.Popen([str(PYTHON), "-m", "amharic_s2s.api", "--repo", str(repo),
                                   "--host", "127.0.0.1", "--port", str(PORT)],
                                  env=env, stdout=log, stderr=subprocess.STDOUT,start_new_session=True)
    print("Started real model server. Run cell 5; it will wait for readiness.")

## Cell 5 — Wait for Ready, then speak the bank questions
Wait for **Ready** and **Bank document search ready · 41 source files**. Start conversation and wait for Listening. Use the eight core questions below. End conversation → New topic → Start conversation between unrelated topics. Manual reference is disabled in bank mode.

For each answer, first check **Heard**, then the retrieved source, then factual accuracy and speech quality. A wrong transcript is an ASR issue; note it and repeat rather than judging retrieval against a different question. Empty recognition resumes listening automatically. This pilot still needs improvements to microphone recognition and difficult compound-query retrieval.

In [ ]:
sys.path.insert(0, str(repo / "src"))
from amharic_s2s.colab_demo import NotebookBridge, show
if "BRIDGE" not in globals():
    ARCHIVE_DIR = SESSION_DIR/"turns"
    BRIDGE = NotebookBridge(PORT, API_TOKEN, archive_dir=ARCHIVE_DIR)
print("Automatic result archive:", BRIDGE.archive_dir)
deadline = time.monotonic() + 1200
while True:
    if SERVER.poll() is not None:
        print(LOG.read_text(errors="replace")[-10000:])
        raise RuntimeError("Model server stopped. Share the error above; do not repeatedly rerun model loading.")
    result = BRIDGE.request("GET", "/ready", authorization="Bearer " + API_TOKEN)
    state = result.get("body") or {}
    if result["status"] == 200 and state.get("ready"): break
    if state.get("state") == "failed":
        print(LOG.read_text(errors="replace")[-10000:])
        raise RuntimeError("Model runtime failed ("+str(state.get("last_error","unknown"))+ "). Export with cell 6, then stop with cell 7. Do not reload yet.")
    if time.monotonic() > deadline:
        print(LOG.read_text(errors="replace")[-6000:])
        raise RuntimeError("Still not ready after 20 minutes. Inspect the log; do not start another server.")
    print("Waiting for model download/load/warmup...", state.get("state", "starting"), flush=True)
    time.sleep(15)
print("REAL MODELS READY. The access key is filled automatically. Wait for this cell to finish, then click Start conversation.")
show(BRIDGE, repo / "src/amharic_s2s/web")
from IPython.display import display, Markdown
display(Markdown((repo / "docs/bank-demo-test-cases.md").read_text()))

## Cell 6 — Save your review and export
End conversation first. Edit QUALITY_NOTES with the actual observations, then run. The ZIP includes completed and failed turns, source passages, retrieval times, separate model/browser timings and startup logs. No raw microphone recordings or credentials are exported.

In [ ]:
from datetime import datetime, timezone
QUALITY_NOTES="""Greeting:
Apollo definition (Heard / source relevance / answer / speech):
Cardless withdrawal:
Mobile registration:
Lost-card longer answer (missing or repeated speech):
Personal balance (must not invent a balance):
Cooking outside scope:
Thanks:
Optional code / compound question:
Empty recognition or incorrect transcripts:
Other observations:
"""
feedback={"tester":OWNER,"package_sha256":hashlib.sha256(PACKAGE.read_bytes()).hexdigest(),
          "tested_at_utc":datetime.now(timezone.utc).isoformat(),"notes":QUALITY_NOTES}
stamp=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
destination=OUTPUT/("bank-demo-"+stamp+"-"+secrets.token_hex(3))
if "BRIDGE" in globals():
    archive=BRIDGE.export(destination)
else:
    destination.mkdir(parents=True,exist_ok=False)
    archive=destination.with_suffix(".zip")
    with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED):pass
(destination/"feedback.json").write_text(json.dumps(feedback,ensure_ascii=False,indent=2),encoding="utf-8")
(destination/"listening_notes.txt").write_text(QUALITY_NOTES,encoding="utf-8")
if "LOG" in globals() and LOG.exists():shutil.copy2(LOG,destination/"api-server.log")
if "SESSION_DIR" in globals():
    for name in ("engine.json","resident_models.json","llama-server.log"):
        path=SESSION_DIR/"engine"/name
        if path.exists():shutil.copy2(path,destination/name)
for name in ("configs/llm_engine_comparison.json","configs/runtime_instructions.json"):
    shutil.copy2(repo/name,destination/Path(name).name)
for source,target in ((repo/"knowledge/bank_v2/build_report.json","corpus_build_report.json"),
                      (OUTPUT/"retrieval_preflight.json","retrieval_preflight.json"),
                      (repo/"configs/bank_demo_cases.json","bank_demo_cases.json")):
    if source.exists():shutil.copy2(source,destination/target)
# Rebuild once so the ZIP has no duplicate entries and includes the listening notes.
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    for path in sorted(destination.rglob("*")):
        if path.is_file():z.write(path,str(Path(destination.name)/path.relative_to(destination)))
print("RESULT ZIP:",archive)
from google.colab import files
files.download(str(archive))
print("Confirm the ZIP download, then run cell 7. Send the downloaded ZIP or extracted folder path.")

## Cell 7 — Stop both servers, then release the GPU runtime

In [ ]:
import signal
if "SERVER" in globals():
    # Both processes belong to the dedicated group created in cell 4.
    try:os.killpg(SERVER.pid,signal.SIGTERM)
    except ProcessLookupError:pass
    try:SERVER.wait(timeout=30)
    except subprocess.TimeoutExpired:pass
    try:os.killpg(SERVER.pid,signal.SIGKILL)
    except ProcessLookupError:pass
    SERVER.wait()
print("Notebook-owned servers stopped. Use Runtime → Disconnect and delete runtime.")